# Reaction-Diffusion Laboratory: saved graphs

**Save graph** copied this notebook next to the data it draws:

| File | Contents |
|---|---|
| `metadata.json` | model, mode, variables, panels |
| `parameters.txt` | all parameters at the time of saving |
| `solution.csv` | `panel`, `x` and one column per variable |
| `spatial_profiles.csv` | `panel`, `x` and one column per spatial profile (models with profiles only) |
| `series_*.csv` | series results (series mode only) |
| `simulation.*`, `series.*` | the graphs drawn by the application |

It needs `pandas` and `matplotlib`. Run it from this folder.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

folder = Path(".")
meta = json.loads((folder / "metadata.json").read_text(encoding="utf-8"))
panels = meta["panels"]

print(f"{meta['model']}   ({meta['boundary']})")
print(f"Saved {meta['saved']} in {meta['mode']} mode, {len(panels)} panel(s)")

## Parameters

In [ ]:
print((folder / "parameters.txt").read_text(encoding="utf-8"))

## Solution

One row per variable, then one per spatial profile; one column per panel, as in the application.

In [ ]:
solution = pd.read_csv(folder / "solution.csv")
profiles_file = folder / "spatial_profiles.csv"
profiles = pd.read_csv(profiles_file) if profiles_file.exists() else None
solution.head()

In [ ]:
rows = [(name, solution, dict(color="C0", linewidth=2)) for name in meta["variables"]]
rows += [(name, profiles, dict(color="red", linestyle="--", linewidth=2)) for name in meta["spatial_profiles"]]

fig, axes = plt.subplots(
    len(rows),
    len(panels),
    figsize=(3 + 4 * len(panels), 2.4 * len(rows)),
    sharey="row",
    squeeze=False,
    gridspec_kw={"width_ratios": [panel["length"] for panel in panels]},
)

for column, panel in enumerate(panels):
    for row, (name, table, style) in enumerate(rows):
        data = table[table["panel"] == panel["panel"]]
        ax = axes[row, column]
        ax.plot(data["x"], data[name], **style)
        ax.set_xlim(0, panel["length"])
        ax.set_title(name if row > 0 else "", fontsize=10)
        if column == 0:
            ax.set_ylabel(name)
        if row == len(rows) - 1:
            ax.set_xlabel("x")
    status = panel["status"].replace("\n", "   ")
    axes[0, column].set_title(f"Panel {panel['panel']}   {status}" if len(panels) > 1 else status, fontsize=9)

fig.suptitle(f"{meta['model']}   ({meta['boundary']})")
fig.tight_layout()
plt.show()

## Series results

Only when the graphs were saved in series mode. In `series_configurations.csv` the head positions are normalised to the panel (0 to 1); the not converged runs of each panel are in `metadata.json`.

In [ ]:
series = meta["mode"] == "series"

if series:
    locations = pd.read_csv(folder / "series_head_locations.csv")
    head_counts = pd.read_csv(folder / "series_head_counts.csv")
    configurations = pd.read_csv(folder / "series_configurations.csv", dtype={"label": str, "positions": str})
    patterns = pd.read_csv(folder / "series_patterns.csv")
    print(f"{meta['completed_runs']} completed runs, head variable {meta['head_variable']}")
else:
    print("Saved in simulation mode: there are no series results.")

In [ ]:
PALETTE = ["#64748b", "#2563eb", "#7c3aed", "#db2777", "#ea580c", "#16a34a", "#0891b2"]

if series:
    fig, axes = plt.subplots(
        4,
        len(panels),
        figsize=(1 + 5 * len(panels), 15),
        squeeze=False,
        gridspec_kw={"height_ratios": [1, 1, 1.4, 1]},
    )

    for column, panel in enumerate(panels):
        number = panel["panel"]
        prefix = f"Panel {number}: " if len(panels) > 1 else ""
        not_converged = panel.get("not_converged", 0)

        # Head locations: one bar per grid node.
        ax = axes[0, column]
        data = locations[locations["panel"] == number]
        width = data["x"].diff().median() if len(data) > 1 else panel["length"]
        ax.bar(data["x"], data["count"], width=width, color="#2563eb")
        ax.set_xlim(0, panel["length"])
        ax.set_title(prefix + "Head locations")
        ax.set_xlabel("x")

        # Number of heads, with the not converged runs last.
        ax = axes[1, column]
        data = head_counts[head_counts["panel"] == number]
        labels = [str(heads) for heads in data["heads"]] + ["Not converged"]
        counts = list(data["count"]) + [not_converged]
        ax.bar(labels, counts, color=["#7c3aed"] * len(data) + ["#dc2626"])
        ax.set_title(prefix + "Number of heads")
        ax.set_ylabel("runs")

        # Head configurations, coloured by the number of heads.
        ax = axes[2, column]
        data = configurations[configurations["panel"] == number]
        labels = list(data["label"]) + ["Not converged"]
        counts = list(data["count"]) + [not_converged]
        colors = [PALETTE[min(heads, len(PALETTE) - 1)] for heads in data["heads"]] + ["#dc2626"]
        ax.barh(range(len(labels)), counts, color=colors)
        ax.set_yticks(range(len(labels)), labels, fontfamily="monospace")
        ax.invert_yaxis()
        ax.set_title(prefix + "Head configurations")
        ax.set_xlabel("runs")

        # Final profiles of the head variable: converged in blue, not converged in red.
        ax = axes[3, column]
        data = patterns[patterns["panel"] == number]
        for (run, converged), profile in data.groupby(["run", "converged"]):
            ax.plot(
                profile["x"],
                profile["value"],
                linewidth=0.8,
                color="#2563eb" if converged else "#dc2626",
                alpha=0.28 if converged else 0.55,
            )
        ax.set_xlim(0, panel["length"])
        ax.set_title(prefix + f"Patterns ({meta['head_variable']})")
        ax.set_xlabel("x")

    fig.suptitle(f"{meta['model']}   series, {meta['completed_runs']} runs")
    fig.tight_layout()
    plt.show()